Generate the dataset (uses `src/sample_data.py`)

**Edit `PROJECT_DIR`.**

In [ ]:
# !pip install torch

# import torch
# print(torch.cuda.is_available())

# from google.colab import drive
# drive.mount('/content/drive')

# %cd /content/drive/MyDrive/DSNSFcomp/nsf-fmrg-data-challenge

In [ ]:
%pip install tensorflow scikit-learn

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# !important! make the `sample_data.py` file into folder `nsf-fmrg-data-challenge/src` of the original github repo.
# Then, put the direction of `nsf-fmrg-data-challenge` folder on your device below :)
# PROJECT_DIR = Path('/Users/jian/MyProjects/NSF-Future-challenge/nsf-fmrg-data-challenge')

PROJECT_DIR = Path.cwd().parent

print('PROJECT_DIR =', PROJECT_DIR)
# ============================================================

In [ ]:
sys.path.append(str(PROJECT_DIR / 'src'))
from sample_data import SampleGenerator, TRACK_IDS

In [ ]:
gen = SampleGenerator(PROJECT_DIR)
print('tracks:', TRACK_IDS, '| grid:', gen.frame_grid()[[0, 1, -1]], 'mm (400 frame centres)')

In [ ]:
# one sample on demand (given a track and an x)
s = gen.make_sample(8, 60.1)
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in s.items()})

## Width labels over the whole grid

`build_labels()` sweeps every frame-centre x for all tracks and returns `width_mm` + a `quality`
flag (False where x is outside a track's height coverage or an edge is clipped). 

In [ ]:
# Build geometry-based labels from actual height values and save them separately from legacy labels.
# Use stable 0.3-mm profiles, a track-calibrated threshold, and final QA.
# model_width_mm is a median-smoothed local descriptor; raw width_mm is preserved.
labels = gen.build_labels(minimum_confidence=0.70, continuity_tolerance_fraction=0.25)
out = PROJECT_DIR / "processed_data" / "labels_geometry_width.npz"
np.savez(out, **labels)
for track_id in TRACK_IDS:
    m = (labels["track_id"] == track_id) & labels["quality"]
    widths = labels["width_mm"][m] * 1000
    model_widths = labels["model_width_mm"][m] * 1000
    print(f"Track {track_id}: accepted={m.sum()}/{(labels['track_id'] == track_id).sum()}",
          f"raw_median_um={np.median(widths) if len(widths) else np.nan:.1f}",
          f"raw_std_um={np.std(widths) if len(widths) else np.nan:.1f}",
          f"model_std_um={np.std(model_widths) if len(model_widths) else np.nan:.1f}")
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, track_id in zip(axes.flat, TRACK_IDS):
    m = (labels["track_id"] == track_id) & labels["quality"]
    ax.plot(labels["x_mm"][m], labels["width_mm"][m] * 1000, ".", alpha=.35, label="raw boundary width")
    ax.plot(labels["x_mm"][m], labels["model_width_mm"][m] * 1000, "-", lw=1.5, label="median-smoothed target")
    ax.set(title=f"Track {track_id}", ylabel="width (µm)"); ax.grid(alpha=.3)
axes[1,0].set_xlabel("x (mm)"); axes[1,1].set_xlabel("x (mm)")
axes[0,0].legend(); fig.suptitle("Geometry-label QA: raw versus model target"); fig.tight_layout(); plt.show()
print("Saved geometry labels to:", out)


In [ ]:
## Visualization of width

In [ ]:
# Visual QA for actual-height geometry labels. Reject uncertain profiles before ML.
import pandas as pd
qa_x = [30.1, 60.1, 90.1]
fig, axes = plt.subplots(4, 3, figsize=(15, 12), sharex=True)
qa_rows = []
for row, track_id in enumerate(TRACK_IDS):
    for col, x_mm in enumerate(qa_x):
        result = gen.width_at(track_id, x_mm)
        ax = axes[row, col]
        ax.plot(result["y_mm"], result["residual_um"], lw=1, color="tab:blue")
        ax.axhline(0, color="black", lw=.8)
        if result["polarity"]:
            ax.axhline(result["polarity"] * result["threshold_um"], color="tab:orange", ls="--", lw=.8)
        ax.fill_between(result["y_mm"], result["residual_um"], 0, where=result["feature_mask"], alpha=.25, color="tab:green")
        ax.set_title(f"T{track_id:02d}, x={x_mm:.1f}: {'ACCEPT' if result['quality'] else 'REJECT'}",
                     color="tab:green" if result["quality"] else "tab:red")
        if np.isfinite(result["left_boundary_mm"]):
            ax.axvline(result["left_boundary_mm"], color="purple", ls="-.", lw=1)
            ax.axvline(result["right_boundary_mm"], color="purple", ls="-.", lw=1)
        ax.set(xlabel="cross-track y (mm)", ylabel="height residual (µm)")
        ax.grid(alpha=.3)
        qa_rows.append({"track_id":track_id, "x_mm":x_mm, "quality":result["quality"],
                        "width_um":result["width_mm"]*1000, "left_mm":result["left_boundary_mm"],
                        "right_mm":result["right_boundary_mm"], "peak_um":result["peak_height_um"],
                        "threshold_um":result["threshold_um"], "confidence":result["confidence"],
                        "rejection_reason":result["rejection_reason"]})
fig.suptitle("Height-derived width QA: profile, threshold, and segmented deposited feature", y=1.01)
fig.tight_layout(); plt.show()
display(pd.DataFrame(qa_rows))


In [ ]:
# Load validated geometry labels and leakage-safe SEM samples for all four tracks.
import pandas as pd
SEM_TRACKS, TRAIN_TRACKS, VALIDATION_TRACK, TEST_TRACK = [8,10,14,21], [8,10], 14, 21
MIN_ACCEPTED_LABELS_BY_TRACK = {8: 100, 10: 100, 14: 50, 21: 11}
label_path = PROJECT_DIR / "processed_data" / "labels_geometry_width.npz"
labels = np.load(label_path, allow_pickle=False)
required_labels = {"track_id","x_mm","width_mm","model_width_mm","quality","left_boundary_mm","right_boundary_mm","confidence"}
missing = required_labels.difference(labels.files)
if missing: raise KeyError(f"Missing geometry-label arrays: {sorted(missing)}")
for track_id in SEM_TRACKS:
    count = int(((labels["track_id"] == track_id) & labels["quality"]).sum())
    if count < MIN_ACCEPTED_LABELS_BY_TRACK[track_id]:
        raise RuntimeError(f"Track {track_id} has only {count} accepted geometry labels (minimum: {MIN_ACCEPTED_LABELS_BY_TRACK[track_id]}). Review the QA plots and calibrate the height-boundary rule before training.")
label_ids = np.array([f"T{int(t):02d}_X{float(x):05.1f}" for t,x in zip(labels["track_id"],labels["x_mm"])])
label_index = {sample_id:i for i,sample_id in enumerate(label_ids)}
sem_by_track = {}
for track_id in SEM_TRACKS:
    path = PROJECT_DIR / "Samples" / "SEM Samples" / f"SEM_samples_Track_{track_id:02d}.npz"
    with np.load(path, allow_pickle=False) as data:
        required = {"sample_ids","track_ids","x_mm","raw_slices","substrate_only_slices","substrate_masks"}
        if required.difference(data.files): raise KeyError(f"Invalid SEM file: {path}")
        reconstructed = np.where(data["substrate_masks"].astype(bool), data["raw_slices"], 0)
        if not np.array_equal(reconstructed.astype(data["substrate_only_slices"].dtype), data["substrate_only_slices"]):
            raise AssertionError(f"Safe SEM reconstruction failed for Track {track_id}")
        sem_by_track[track_id] = {"ids":data["sample_ids"].astype(str), "x":data["x_mm"].astype(float),
                                  "image":data["substrate_only_slices"].copy(), "mask":data["substrate_masks"].astype(bool).copy()}
print("Geometry-label and safe-SEM inputs loaded.")


In [ ]:
# Align geometry labels, safe SEM, and track metadata strictly by sample_id.
parts = []
for track_id in SEM_TRACKS:
    eligible = label_ids[(labels["track_id"] == track_id) & labels["quality"]]
    data = sem_by_track[track_id]; ids = np.intersect1d(data["ids"], eligible)
    if not len(ids): raise ValueError(f"No aligned accepted labels for Track {track_id}")
    lookup = {s:i for i,s in enumerate(data["ids"])}
    si = np.array([lookup[s] for s in ids]); li = np.array([label_index[s] for s in ids])
    parts.append((ids, np.full(len(ids),track_id), data["x"][si], data["image"][si], data["mask"][si], labels["model_width_mm"][li]*1000))
aligned_sample_ids = np.concatenate([p[0] for p in parts]); aligned_track_ids = np.concatenate([p[1] for p in parts])
aligned_x_mm = np.concatenate([p[2] for p in parts]); aligned_sem_images = np.concatenate([p[3] for p in parts])
aligned_sem_masks = np.concatenate([p[4] for p in parts]); aligned_width_um = np.concatenate([p[5] for p in parts]).astype(np.float32)
print(pd.DataFrame({"track":aligned_track_ids}).value_counts().sort_index())


In [ ]:
# Load the raw five-frame thermal sequence.  Do not collapse time into summary maps.
k = 2
aligned_thermal = np.stack([gen.thermal_tensor(int(t), float(x), k=k) for t,x in zip(aligned_track_ids, aligned_x_mm)])
print("Raw thermal tensors:", aligned_thermal.shape)


In [ ]:
# Train: Tracks 8/10. Validation: Track 14. Test: Track 21 only after selection.
import tensorflow as tf
from sklearn.metrics import mean_absolute_error
sem_input=np.stack([aligned_sem_images.astype(np.float32)/255, aligned_sem_masks.astype(np.float32)],axis=-1)
y_um=aligned_width_um
train_idx=np.flatnonzero(np.isin(aligned_track_ids,TRAIN_TRACKS)); val_idx=np.flatnonzero(aligned_track_ids==VALIDATION_TRACK); test_idx=np.flatnonzero(aligned_track_ids==TEST_TRACK)
thermal_scale=max(float(aligned_thermal[train_idx].max()),1.0)
# (sample, time, height, width, channel): full sequence enters the learned encoder.
thermal_sequence=(aligned_thermal.astype(np.float32) / thermal_scale)[...,None]
y_mean=float(y_um[train_idx].mean()); y_std=float(y_um[train_idx].std()); y_z=(y_um-y_mean)/y_std
x_mean=float(aligned_x_mm[train_idx].mean()); x_std=float(aligned_x_mm[train_idx].std()); x_input=((aligned_x_mm-x_mean)/x_std).astype(np.float32)[:,None]
X_sem_train,X_sem_val=sem_input[train_idx],sem_input[val_idx]; X_th_train,X_th_val=thermal_sequence[train_idx],thermal_sequence[val_idx]; X_x_train,X_x_val=x_input[train_idx],x_input[val_idx]
sem_layer=tf.keras.Input(sem_input.shape[1:],name="sem_input"); a=tf.keras.layers.Conv2D(8,3,activation="relu",padding="same")(sem_layer); a=tf.keras.layers.MaxPool2D(2)(a); a=tf.keras.layers.Conv2D(16,3,activation="relu",padding="same")(a); a=tf.keras.layers.GlobalAveragePooling2D()(a)
# Shared spatial CNN per frame, then a GRU over the ordered five-frame sequence.
th_layer=tf.keras.Input(thermal_sequence.shape[1:],name="thermal_sequence")
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(8,5,strides=4,activation="relu",padding="same"))(th_layer)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(16,3,strides=2,activation="relu",padding="same"))(b)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(24,3,strides=2,activation="relu",padding="same"))(b)
b=tf.keras.layers.TimeDistributed(tf.keras.layers.GlobalAveragePooling2D())(b)
b=tf.keras.layers.GRU(32,dropout=.10)(b)
x_layer=tf.keras.Input((1,),name="x_input"); c=tf.keras.layers.Dense(8,activation="relu")(x_layer)
z=tf.keras.layers.Concatenate()([a,b,c]); z=tf.keras.layers.Dense(32,activation="relu")(z); z=tf.keras.layers.Dropout(.2)(z); out=tf.keras.layers.Dense(1)(z)
model=tf.keras.Model([sem_layer,th_layer,x_layer],out); model.compile(tf.keras.optimizers.Adam(5e-4),loss=tf.keras.losses.Huber(delta=1.0),metrics=["mae"])
checkpoint=PROJECT_DIR/"processed_data"/"best_geometry_width_sequence_model.keras"
history=model.fit({"sem_input":X_sem_train,"thermal_sequence":X_th_train,"x_input":X_x_train},y_z[train_idx],validation_data=({"sem_input":X_sem_val,"thermal_sequence":X_th_val,"x_input":X_x_val},y_z[val_idx]),epochs=150,batch_size=8,callbacks=[tf.keras.callbacks.ModelCheckpoint(checkpoint,monitor="val_loss",save_best_only=True),tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=15,restore_best_weights=True)],verbose=2)


In [ ]:
print("Thermal sequence used by the model:", thermal_sequence.shape)
print("Train-only normalization scale:", thermal_scale)
print("Sequence mean/std:", float(thermal_sequence.mean()), float(thermal_sequence.std()))


In [ ]:
pred_val=model.predict({"sem_input":X_sem_val,"thermal_sequence":X_th_val,"x_input":X_x_val})[:,0]*y_std+y_mean
true_val=y_um[val_idx]; baseline_val=np.full_like(true_val,y_mean)
print(f"Track-{VALIDATION_TRACK} MAE model / baseline:", mean_absolute_error(true_val,pred_val), mean_absolute_error(true_val,baseline_val))
plt.scatter(true_val,pred_val,alpha=.6); lim=[min(true_val.min(),pred_val.min()),max(true_val.max(),pred_val.max())]; plt.plot(lim,lim,"r--"); plt.xlabel("true geometry width (µm)"); plt.ylabel("predicted width (µm)"); plt.grid(alpha=.3); plt.show()


## Width labels over the whole grid

`build_labels()` sweeps every frame-centre x for all tracks and returns `width_mm` + a `quality`
flag (False where x is outside a track's height coverage or an edge is clipped). 

In [ ]:
# Inspect the configured validation-track errors only; do not use Track 21 for model selection.
print(f"Validation table: Track {VALIDATION_TRACK}; samples={len(val_idx)}")
validation_table=pd.DataFrame({"sample_id":aligned_sample_ids[val_idx],"x_mm":aligned_x_mm[val_idx],"true_width_um":true_val,"predicted_width_um":pred_val,"error_um":pred_val-true_val})
validation_table["abs_error_um"]=validation_table.error_um.abs(); display(validation_table.sort_values("abs_error_um",ascending=False).head(10))


In [ ]:
# Final evaluation on the untouched Track 21.
pred_test=model.predict({"sem_input":sem_input[test_idx],"thermal_sequence":thermal_sequence[test_idx],"x_input":x_input[test_idx]})[:,0]*y_std+y_mean
true_test=y_um[test_idx]; baseline_test=np.full_like(true_test,y_mean)
print("Track-21 MAE model / baseline:",mean_absolute_error(true_test,pred_test),mean_absolute_error(true_test,baseline_test))
result=pd.DataFrame({"sample_id":aligned_sample_ids[test_idx],"x_mm":aligned_x_mm[test_idx],"true_width_um":true_test,"predicted_width_um":pred_test,"error_um":pred_test-true_test}); result["abs_error_um"]=result.error_um.abs(); display(result.sort_values("abs_error_um",ascending=False).head(10))
